# N2H validation in mammalian cells

In [1]:
import glob
import xlrd
import pandas as pd
import numpy as np
import os

In [2]:
inputs_dir = "../1_inputs/N2H_validation"
allele_inputs = "../../../1_allele_collection/1_inputs"
outputs_dir = "../3_outputs/N2H_validation/mammalian_N2H"

nlr_detection_threshold = 2.23

## Wild type pairwise test

In [22]:
import warnings
warnings.simplefilter("ignore")

sheets = [f'Feuil{i}' for i in range (1, 22)]

wt_res = {'Exp1_1':dict(), 'Exp1_2':dict(), 'Exp2_1':dict(), 'Exp2_2':dict()}

for file_name in glob.glob('/Users/jewald/repos/2025_laval_submitted/2_individual_assay_results/ppi/1_inputs/N2H_validation/mammalian_N2H/mN2H_WTPWT/data/*'):
     if 'N2H' in file_name:
        file_name = file_name.replace('~$','')
        xp = f'{file_name.split("_")[-1][:-7]}_{file_name.split("_")[-1][-6]}'

        for sheet in sheets:

            df =  pd.read_excel(file_name, sheet_name=sheet)
            df.rename(columns={c:c.lower() for c in df}, inplace=True)

            for dest_pla, dest_well, RawData, NA in zip(df.dest_pla, df['dest_well.1'], df['raw data'], df['unnamed: 13']):
                if NA != 1:
                    wt_res[xp][(dest_pla,dest_well)] = RawData
                else:
                    wt_res[xp][(dest_pla,dest_well)] = np.nan

In [23]:
# Read in the plate layout file
col = ['ad_orf_id', 'ad_symbol', 'db_orf_id',
       'db_symbol','variant_id', 'retest_pla', 'retest_pos', 
       'standard_batch']
retest_df = pd.read_csv(f'{inputs_dir}/mammalian_N2H/mapping_mn2h.csv', low_memory=False)
wt_df = retest_df[retest_df.standard_batch == 'N2HVUSWT'][col]

In [24]:
for xp in wt_res:
    wt_df[xp] = [wt_res[xp][(pla, pos)] for pla, pos in zip(wt_df.retest_pla, wt_df.retest_pos)]
    
wt_df.rename(columns={c:c.replace('ad','N1').replace('db','N2') for c in wt_df.columns}, inplace=True)
wt_df.rename(columns={'Exp1_1':'mN2H_xp1r1','Exp1_2':'mN2H_xp1r2','Exp2_1':'mN2H_xp2r1','Exp2_2':'mN2H_xp2r2'}, inplace=True)

wt_df['xp1'] = [np.mean([e for e in [r1,r2] if e == e]) if r1/r2 > 0.2 and r1/r2 < 5 else np.nan for r1,r2 in zip(wt_df.mN2H_xp1r1, wt_df.mN2H_xp1r2)]
wt_df['xp2'] = [np.mean([e for e in [r1,r2] if e == e]) if r1/r2 > 0.2 and r1/r2 < 5 else np.nan for r1,r2 in zip(wt_df.mN2H_xp2r1, wt_df.mN2H_xp2r2)]

In [25]:
# calculate normalized luminescence ratio
empty_N1_xp1_d = dict(zip(wt_df[wt_df.variant_id == 'empty_N1'].N2_orf_id, wt_df[wt_df.variant_id == 'empty_N1'].xp1))
empty_N2_xp1_d = dict(zip(wt_df[wt_df.variant_id == 'empty_N2'].N1_orf_id, wt_df[wt_df.variant_id == 'empty_N2'].xp1))

empty_N1_xp2_d = dict(zip(wt_df[wt_df.variant_id == 'empty_N1'].N2_orf_id, wt_df[wt_df.variant_id == 'empty_N1'].xp2))
empty_N2_xp2_d = dict(zip(wt_df[wt_df.variant_id == 'empty_N2'].N1_orf_id, wt_df[wt_df.variant_id == 'empty_N2'].xp2))

normalized_wt_df = wt_df[wt_df.variant_id.isin(['wt','RRS','PRS', 'allele'])].copy()

normalized_wt_df['empty_N1_xp1'] = normalized_wt_df.apply(lambda x: empty_N1_xp1_d[x.N2_orf_id], axis=1)
normalized_wt_df['empty_N2_xp1'] = normalized_wt_df.N1_orf_id.map(empty_N2_xp1_d)

normalized_wt_df['empty_N1_xp2'] = normalized_wt_df.apply(lambda x: empty_N1_xp2_d[x.N2_orf_id], axis=1)
normalized_wt_df['empty_N2_xp2'] = normalized_wt_df.N1_orf_id.map(empty_N2_xp2_d)

normalized_wt_df['NLR_xp1'] = [np.log2(N1N2/max([N1F2,N2F1])) for N1N2, N1F2, N2F1 in zip(normalized_wt_df.xp1, normalized_wt_df.empty_N1_xp1, normalized_wt_df.empty_N2_xp1)]
normalized_wt_df['NLR_xp2'] = [np.log2(N1N2/max([N1F2,N2F1])) for N1N2, N1F2, N2F1 in zip(normalized_wt_df.xp2, normalized_wt_df.empty_N1_xp2, normalized_wt_df.empty_N2_xp2)]

df = normalized_wt_df[normalized_wt_df.variant_id == 'RRS']
normalized_wt_df['zNLR1'] = normalized_wt_df.NLR_xp1.apply(lambda x: (x-np.mean(df.NLR_xp1))/np.std(df.NLR_xp1))
normalized_wt_df['zNLR2'] = normalized_wt_df.NLR_xp2.apply(lambda x: (x-np.mean(df.NLR_xp2))/np.std(df.NLR_xp2))

normalized_wt_df['NLR'] = normalized_wt_df.apply(lambda x: np.mean([x.NLR_xp1, x.NLR_xp2]), axis=1)

normalized_wt_df['pair'] = [tuple(sorted([n1,n2])) for n1,n2 in zip(normalized_wt_df.N1_symbol, normalized_wt_df.N2_symbol)]

In [ ]:
# Filter out alleles that were not sequence confirmed

In [26]:
# Count the number detected by gene for different thresholds
normalized_wt_df['detected_0'] = normalized_wt_df.NLR.apply(lambda x: x > sorted(normalized_wt_df[normalized_wt_df.variant_id == 'RRS'].NLR.dropna().tolist())[-1])
normalized_wt_df['detected_1'] = normalized_wt_df.NLR.apply(lambda x: x > sorted(normalized_wt_df[normalized_wt_df.variant_id == 'RRS'].NLR.dropna().tolist())[-2])
normalized_wt_df['detected_2'] = normalized_wt_df.NLR.apply(lambda x: x > sorted(normalized_wt_df[normalized_wt_df.variant_id == 'RRS'].NLR.dropna().tolist())[-3])
normalized_wt_df['detected_3'] = normalized_wt_df.NLR.apply(lambda x: x > sorted(normalized_wt_df[normalized_wt_df.variant_id == 'RRS'].NLR.dropna().tolist())[-4])

by_gene_df = pd.DataFrame([pd.Series(dict(normalized_wt_df[(normalized_wt_df.variant_id == 'wt')].N2_symbol.value_counts())),
                           pd.Series(normalized_wt_df[normalized_wt_df.detected_0 & (normalized_wt_df.variant_id == 'wt')].N2_symbol.value_counts()),
                           pd.Series(normalized_wt_df[normalized_wt_df.detected_1 & (normalized_wt_df.variant_id == 'wt')].N2_symbol.value_counts()),
                           pd.Series(normalized_wt_df[normalized_wt_df.detected_2 & (normalized_wt_df.variant_id == 'wt')].N2_symbol.value_counts()),
                           pd.Series(normalized_wt_df[normalized_wt_df.detected_3 & (normalized_wt_df.variant_id == 'wt')].N2_symbol.value_counts())
                           
                          ]).T

by_gene_df.columns = ['y2h', 'mn2h_0','mn2h_1','mn2h_2', 'mn2h_3']
by_gene_df.fillna(0, inplace=True)
by_gene_df.sort_values(by='mn2h_0', ascending=False, inplace=True)

In [27]:
# Write out files
normalized_wt_df.to_csv(f'{outputs_dir}/mN2H_wtpwt_raw.csv')
by_gene_df.to_csv(f'{outputs_dir}/mN2H_wtpwt_counts.csv')

normalized_wt_df[(normalized_wt_df.detected_3 == True) & (normalized_wt_df.variant_id == 'wt')].to_csv(f'{outputs_dir}/mN2H_wtpwt_positive_PPIs.csv') 

## Allele pairwise test

In [3]:
# Dictionaries to map plate layouts
xp1_mapping_d = {i: ((i - 1) // 3 + 1, (i - 1) % 3 + 1) for i in range(1, 70)}

xp2_mapping_d = {
    1: (1,1),  2: (1,2),  3: (1,3),  4: (2,1),  5: (2,2),  6: (2,3),  
    7: (3,1),  8: (3,2),  9: (3,3), 10: (4,1), 11: (4,2), 12: (4,3),  
    13: (7,1), 14: (7,2), 15: (7,3), 16: (6,1), 17: (6,2), 18: (6,3),  
    19: (5,1), 20: (5,2), 21: (5,3), 22: (9,1), 23: (9,2), 24: (9,3),  
    25: (10,1), 26: (10,2), 27: (10,3), 28: (11,1), 29: (11,2), 30: (11,3),  
    31: (12,1), 32: (12,2), 33: (12,3), 34: (13,1), 35: (13,2), 36: (13,3),  
    37: (14,1), 38: (14,2), 39: (14,3), 40: (15,1), 41: (15,2), 42: (15,3),  
    43: (16,1), 44: (16,2), 45: (16,3), 46: (17,1), 47: (17,2), 48: (17,3),  
    49: (18,1), 50: (18,2), 51: (18,3), 52: (19,1), 53: (19,2), 54: (19,3),  
    55: (20,1), 56: (20,2), 57: (20,3), 58: (21,1), 59: (21,2), 60: (21,3),  
    61: (22,1), 62: (22,2), 63: (22,3), 64: (23,1), 65: (23,2), 66: (23,3),  
    67: (8,1), 68: (8,2), 69: (8,3)
}

In [4]:
# Load in raw data for allele pairwise test (apwt) from the mammalian N2H validation assay
res = {'xp1':{1:{},2:{},3:{}},
        'xp2':{1:{},2:{},3:{}}
        }

for rep_id in ['1','2']:
    for i,elem in enumerate(glob.glob(f'{inputs_dir}/mammalian_N2H/mN2H_APWT/N2H_raw_replicat0{rep_id}/*')):
        pla = int(elem.split('-')[-1][1:-4])
        retest_pla, repeat = xp1_mapping_d[pla] if rep_id == '1' else xp2_mapping_d[pla]

        wb = xlrd.open_workbook(elem, encoding_override='ascii')  #ascii  #'iso-8859-1'
        df = pd.read_excel(wb, skiprows=24, skipfooter=0)
        if len(df) > 8:
            df = pd.read_excel(wb, skiprows=35, skipfooter=10)

        df.rename(columns={c:i+1 for i,c in enumerate(df.columns)}, inplace=True)

        value_d = dict()
        for l, (i, row) in zip(['A','B','C','D','E','F','G','H'], df.iterrows()):
            for i, value in enumerate(row):
                key = (retest_pla,f'{l}{str(i+1).zfill(2)}')

                res[f'xp{rep_id}'][repeat][key] = value

In [5]:
# Read in the plate layout file
retest_df = pd.read_csv(f'{inputs_dir}/mammalian_N2H/mapping_mn2h.csv', low_memory=False)
retest_df = retest_df[retest_df.standard_batch == 'mN2Hapwtv2'].copy()
retest_df.rename(columns={c:c.replace('ad','N1').replace('db','N2') for c in retest_df.columns},inplace=True)

In [6]:
# Calculate normalized luminescence ratio
for xp in ['xp1','xp2']:
    for rep in [1,2,3]:
        retest_df[f'mN2H_{xp}r{rep}'] = [res[xp][rep][pla,pos] if (pla,pos) in res[xp][rep] else np.nan for pla,pos in zip(retest_df.retest_pla, retest_df.retest_pos)]

#discard measure if signal is 20% of those of 2 other replicates (it's never 5 fold more than the 2 other measures)    
retest_df['mN2H_xp1r1'] = [np.nan if r1/r2<0.2 and r1/r3<0.2 else r1 for r1,r2,r3 in zip(retest_df.mN2H_xp1r1, retest_df.mN2H_xp1r2, retest_df.mN2H_xp1r3)]
retest_df['mN2H_xp1r2'] = [np.nan if r2/r1<0.2 and r2/r3<0.2 else r2 for r1,r2,r3 in zip(retest_df.mN2H_xp1r1, retest_df.mN2H_xp1r2, retest_df.mN2H_xp1r3)]
retest_df['mN2H_xp1r3'] = [np.nan if r3/r2<0.2 and r3/r1<0.2 else r3 for r1,r2,r3 in zip(retest_df.mN2H_xp1r1, retest_df.mN2H_xp1r2, retest_df.mN2H_xp1r3)]
retest_df['mN2H_xp2r1'] = [np.nan if r1/r2<0.2 and r1/r3<0.2 else r1 for r1,r2,r3 in zip(retest_df.mN2H_xp2r1, retest_df.mN2H_xp2r2, retest_df.mN2H_xp2r3)]
retest_df['mN2H_xp2r2'] = [np.nan if r2/r1<0.2 and r2/r3<0.2 else r2 for r1,r2,r3 in zip(retest_df.mN2H_xp2r1, retest_df.mN2H_xp2r2, retest_df.mN2H_xp2r3)]
retest_df['mN2H_xp2r3'] = [np.nan if r3/r2<0.2 and r3/r1<0.2 else r3 for r1,r2,r3 in zip(retest_df.mN2H_xp2r1, retest_df.mN2H_xp2r2, retest_df.mN2H_xp2r3)]

xp_errors = {1:{(5,'E03')},
                2:{(5,'E03'),
                (13,'A05'),
                (13,'A06'),
                (13,'D06'),
                (14,'C01'),
                (14,'B04'),
                (14,'E06'),
                (14,'H06'),
                (14,'B08'),
                (14,'A10'),
                }
            }

retest_df['error1'] = [(p,w) in [(pla,well) for pla,well in xp_errors[1]] for p,w in zip(retest_df.retest_pla, retest_df.retest_pos)]
retest_df['error2'] = [(p,w) in [(pla,well) for pla,well in xp_errors[2]] for p,w in zip(retest_df.retest_pla, retest_df.retest_pos)]

retest_df['xp1'] = [np.mean([e for e in [r1,r2,r3] if e == e]) for r1,r2,r3 in zip(retest_df.mN2H_xp1r1, retest_df.mN2H_xp1r2, retest_df.mN2H_xp1r3)]
retest_df['xp2'] = [np.mean([e for e in [r1,r2,r3] if e == e]) for r1,r2,r3 in zip(retest_df.mN2H_xp2r1, retest_df.mN2H_xp2r2, retest_df.mN2H_xp2r3)]

retest_df['xp1'] = [xp1 if not error else np.nan for xp1,error in zip(retest_df.xp1, retest_df.error1)]
retest_df['xp2'] = [xp2 if not error else np.nan for xp2,error in zip(retest_df.xp2, retest_df.error2)]

empty_N1_xp1_d = dict(zip(zip(retest_df[retest_df.variant_id == 'empty_N1'].N2_orf_id,retest_df[retest_df.variant_id == 'empty_N1'].N2_mut_id),retest_df[retest_df.variant_id == 'empty_N1'].xp1))
empty_N2_xp1_d = dict(zip(retest_df[retest_df.variant_id == 'empty_N2'].N1_orf_id, retest_df[retest_df.variant_id == 'empty_N2'].xp1))

empty_N1_xp2_d = dict(zip(zip(retest_df[retest_df.variant_id == 'empty_N1'].N2_orf_id,retest_df[retest_df.variant_id == 'empty_N1'].N2_mut_id),retest_df[retest_df.variant_id == 'empty_N1'].xp2))
empty_N2_xp2_d = dict(zip(retest_df[retest_df.variant_id == 'empty_N2'].N1_orf_id, retest_df[retest_df.variant_id == 'empty_N2'].xp2))

normalized_df = retest_df[retest_df.variant_id.isin(['wt','RRS','PRS', 'allele'])].copy()

normalized_df.drop(columns=['error1','error2','mN2H_xp1r1','mN2H_xp1r2','mN2H_xp1r3','mN2H_xp2r1','mN2H_xp2r2','mN2H_xp2r3'], inplace=True)

normalized_df['empty_N1_xp1'] = normalized_df.apply(lambda x: empty_N1_xp1_d[x.N2_orf_id,x.N2_mut_id], axis=1)
normalized_df['empty_N2_xp1'] = normalized_df.N1_orf_id.map(empty_N2_xp1_d)

normalized_df['empty_N1_xp2'] = normalized_df.apply(lambda x: empty_N1_xp2_d[x.N2_orf_id,x.N2_mut_id], axis=1)
normalized_df['empty_N2_xp2'] = normalized_df.N1_orf_id.map(empty_N2_xp2_d)

normalized_df['max_ctrl_r1'] = [max([en1,en2]) for en1, en2 in zip(normalized_df.empty_N1_xp1, normalized_df.empty_N2_xp1)]
normalized_df['max_ctrl_r2'] = [max([en1,en2]) for en1, en2 in zip(normalized_df.empty_N1_xp2, normalized_df.empty_N2_xp2)]

normalized_df['NLR1'] = [N1N2/max([N1F2,N2F1]) for N1N2, N1F2, N2F1 in zip(normalized_df.xp1, normalized_df.empty_N1_xp1, normalized_df.empty_N2_xp1)]
normalized_df['NLR2'] = [N1N2/max([N1F2,N2F1]) for N1N2, N1F2, N2F1 in zip(normalized_df.xp2, normalized_df.empty_N1_xp2, normalized_df.empty_N2_xp2)]

d = {0:1,1:0}
normalized_df['repeat_ratio'] = [[nlr1,nlr2][i%2]/[nlr1,nlr2][d[(i%2)]] for i,(nlr1, nlr2) in enumerate(zip(normalized_df.NLR1, normalized_df.NLR2))]
normalized_df['repeat_ratio_log2'] = normalized_df.repeat_ratio.apply(np.log2)

normalized_df['avg_NLR'] = [np.mean([nlr1, nlr2]) for nlr1,nlr2 in zip(normalized_df.NLR1, normalized_df.NLR2)]

In [ ]:
# Get perturbation z-scores
allele_df = normalized_df.query("variant_id == 'allele' and ~avg_NLR.isnull()").copy()

df = normalized_df.query("variant_id == 'wt'")
wt_NLR_d = {(r.N2_orf_id, r.N1_orf_id):r.avg_NLR for i,r in df.iterrows()}

allele_df['avg_WT_NLR'] = [wt_NLR_d[N2,N1] for N2,N1 in zip(allele_df.N2_orf_id, allele_df.N1_orf_id)]
allele_df = allele_df[~allele_df.avg_WT_NLR.isnull()].copy() #remove only one allele-WT
allele_df['AWT_ratio'] = allele_df.avg_NLR / allele_df.avg_WT_NLR

df = normalized_df.query("variant_id == 'RRS' and avg_NLR < 800")
l = [np.mean([nlr1,nlr2]) for nlr1,nlr2 in zip(df.NLR1, df.NLR2) if nlr1 == nlr1 and nlr2 == nlr2]
std_rrs, mean_rrs = np.std(l), np.mean(l)
allele_df['z_WT_NLR'] = allele_df.avg_WT_NLR.apply(lambda x: (x-mean_rrs)/std_rrs)
allele_df['WT_detected'] = allele_df.z_WT_NLR.apply(lambda x: x >= nlr_detection_threshold)

std_ratio = np.std(normalized_df.repeat_ratio_log2)
mean_ratio = np.mean(normalized_df.repeat_ratio_log2)
allele_df['z_perturbation'] = allele_df.AWT_ratio.apply(lambda x: (np.log2(x)-mean_ratio)/std_ratio)
allele_df['z_cat'] = ['not_detected' if not wt_detected else 'increase_ppi' if zp>=1.96 else 'decrease_ppi' if zp<=-1.96 else 'no_perturbation' for zp,wt_detected in zip(allele_df.z_perturbation, allele_df.WT_detected)] 

In [ ]:
# Write out final file
allele_df.to_csv(f'{outputs_dir}/mN2H_apwt_allele_df.csv')

## Final figure

In [12]:
#wtpwt_normalized_df = pd.read_csv(f'{outputs_dir}/mN2H_wtpwt_allele_df.csv')
apwt_normalized_df = pd.read_csv(f'{outputs_dir}/mN2H_apwt_allele_df.csv')

keep_cols = ["variant_id", "avg_NLR", "repeat_zscore", "ppi_perturbation_class_04"]

In [ ]:
#limited to WT positive pairs
rrs_mean = np.mean(normalized_df[(normalized_df.variant_id == 'RRS')&(normalized_df.avg_NLR < 200)].avg_NLR)
rrs_std = np.std(normalized_df[(normalized_df.variant_id == 'RRS')&(normalized_df.avg_NLR < 200)].avg_NLR)

threshold = 2.23*rrs_std+rrs_mean
print(threshold)

positive_wt_pairs = set(normalized_df[(normalized_df.variant_id == 'wt')&(normalized_df.avg_NLR >= threshold )].pair)
                            

res_d = {}

for i in range(-8,800):
    trsh = i
    
    rrs_pos, rrs_neg = (normalized_df[(normalized_df.variant_id == 'RRS')].avg_NLR >= trsh).sum(), (normalized_df[(normalized_df.variant_id == 'RRS')].avg_NLR < trsh).sum()
    rrs_lower, rrs_upper = stats.beta.interval(0.6827, rrs_pos + 1, rrs_neg + 1)
    rrs_rate = rrs_pos/(rrs_pos+rrs_neg)
    
    wt_pos = (normalized_df[(normalized_df.variant_id == 'wt')&
                            (normalized_df.pair.isin(positive_wt_pairs))&
                            (normalized_df.avg_NLR >= threshold)&
                            (normalized_df.repeat_zscore == False)
                            ].avg_NLR >= trsh).sum()

    wt_neg = (normalized_df[(normalized_df.variant_id == 'wt')&
                            (normalized_df.pair.isin(positive_wt_pairs))&
                            (normalized_df.avg_NLR >= threshold)&
                            (normalized_df.repeat_zscore == False)].avg_NLR < trsh).sum()

    wt_lower, wt_upper = stats.beta.interval(0.6827, wt_pos + 1, wt_neg + 1)
    wt_rate = wt_pos/(wt_pos+wt_neg)
    res_d[i] = {'trsh':trsh,'rrs_rate':rrs_rate, 'rrs_lower':rrs_lower, 'rrs_upper':rrs_upper,'wt_rate':wt_rate, 'wt_lower':wt_lower, 'wt_upper':wt_upper}
    
    for cat in ['HD-D','HD-P','LD-D','LD-P']:
        pos = ((normalized_df.ppi_perturbation_class_04 == cat)&(normalized_df.avg_NLR >= trsh)&
                 (normalized_df.pair.isin(positive_wt_pairs))&
                 (normalized_df.repeat_zscore == False)).sum()

        neg = ((normalized_df.ppi_perturbation_class_04 == cat)&(normalized_df.avg_NLR < trsh)&
                 (normalized_df.pair.isin(positive_wt_pairs))&
                 (normalized_df.repeat_zscore == False)).sum()

        lower, upper = stats.beta.interval(0.6827, pos + 1, neg + 1)
        rate = pos/(pos+neg) 
        res_d[i][f'{cat}_rate'] = rate
        res_d[i][f'{cat}_lower'] = lower
        res_d[i][f'{cat}_upper'] = upper

rate_df = pd.DataFrame(res_d).T

plt.figure(figsize=(6,6))
plt.title('mN2H Validation - Titration Curve')
cats = ['rrs','wt','HD-D','HD-P','LD-D','LD-P']
cat_d = dict(zip(cats,['RRS','WT','HD-D','HD-P','LD-D','LD-P']))
colors = ['r','orchid','orange','seagreen','gold','gray']

for cat,color in zip(cats,colors):

    ax = sns.lineplot(data=rate_df, x='trsh' , y=f'{cat}_rate', color=color, label=cat_d[cat])
    ax.fill_between(rate_df.trsh, rate_df[f'{cat}_upper'], rate_df[f'{cat}_lower'], alpha=0.2, color=color)

plt.axvline(x=threshold, color='grey', linestyle='--', alpha=0.5)

#plt.xlim((0,7))
plt.ylim((-0.05,1.05))
plt.xlabel('NLR (log2)')
plt.ylabel('Detection rate')
plt.xscale('log', base=2)
# plt.savefig(f'{folder_path}/mN2Hapwt_validation_titration_curve.pdf',  bbox_inches='tight')
# plt.savefig(f'{folder_path}/mN2Hapwt_validation_titration_curve.png', dpi=300,  bbox_inches='tight')
plt.show();

plt.figure(figsize=(4,4))
plt.title('mN2H validation - Detection Rate ')
lower_error = [abs(res_d[int(threshold)][f'{cat}_rate'] - res_d[int(threshold)][f'{cat}_lower'])  for cat in cats]
upper_error = [abs(res_d[int(threshold)][f'{cat}_rate'] - res_d[int(threshold)][f'{cat}_upper'])  for cat in cats]
asymmetric_error = np.array(list(zip(lower_error, upper_error))).T

x = cat_d.values()
y = [e[1] for e in res_d[int(threshold)].items() if 'rate' in e[0]]

plt.bar(x, y, color = colors,alpha=0.7)  #color=['black', 'red', 'green', 'blue', 'cyan']
plt.errorbar(x, y, yerr=asymmetric_error, fmt='.', color = 'black',ecolor = 'black', capsize=5)
plt.ylim((0,1.1))
plt.ylabel('Detection Rate')
plt.xticks(rotation = 45, ha='right')
# plt.savefig(f'{folder_path}/mN2Hapwt_validation_detection_rate.pdf',  bbox_inches='tight')
# plt.savefig(f'{folder_path}/mN2Hapwt_validation_detection_rate.png', dpi=300,  bbox_inches='tight')
plt.show()